In [0]:
-- ============================================================
-- GOLD: Star schema (dims + facts) + enriched business aggregates
-- ============================================================
USE CATALOG f1_project;

-- ============ DIMENSION TABLES ============
-- Dimensions are straight pass-throughs from Silver at this point — but modeled
-- as their own Gold objects because this is the layer BI tools connect to,
-- and because it gives you a clean seam to add slowly-changing-dimension (SCD)
-- logic later without touching Silver.

CREATE OR REPLACE TABLE f1_project.gold.dim_circuits AS
SELECT circuit_id, circuit_ref, name AS circuit_name, location, country, lat, lng, alt
FROM f1_project.silver.circuits_clean;

CREATE OR REPLACE TABLE f1_project.gold.dim_races AS
SELECT r.race_id, r.year, r.round, r.circuit_id, r.name AS race_name,
       r.race_date, r.race_time, c.circuit_name, c.country
FROM f1_project.silver.races_clean r
LEFT JOIN f1_project.gold.dim_circuits c ON r.circuit_id = c.circuit_id;

CREATE OR REPLACE TABLE f1_project.gold.dim_drivers AS
SELECT driver_id, driver_ref, number, code, forename, surname, full_name, dob, nationality,
       DATEDIFF(YEAR, dob, current_date()) AS current_age
FROM f1_project.silver.drivers_clean;

CREATE OR REPLACE TABLE f1_project.gold.dim_constructors AS
SELECT constructor_id, constructor_ref, name AS constructor_name, nationality
FROM f1_project.silver.constructors_clean;

-- ============ FACT TABLES (enriched with dimension keys/attributes) ============

CREATE OR REPLACE TABLE f1_project.gold.fact_results AS
SELECT
  res.result_id, res.race_id, res.driver_id, res.constructor_id,
  res.grid, res.position, res.points, res.laps, res.result_time,
  res.fastest_lap, res.rank, res.fastest_lap_time, res.status,
  ra.year, ra.round, ra.race_name, ra.race_date,
  d.full_name AS driver_name, d.nationality AS driver_nationality,
  co.constructor_name,
  -- enrichment: did the driver gain or lose positions vs. their grid slot?
  (res.grid - res.position) AS positions_gained
FROM f1_project.silver.results_clean res
LEFT JOIN f1_project.gold.dim_races ra ON res.race_id = ra.race_id
LEFT JOIN f1_project.gold.dim_drivers d ON res.driver_id = d.driver_id
LEFT JOIN f1_project.gold.dim_constructors co ON res.constructor_id = co.constructor_id;

CREATE OR REPLACE TABLE f1_project.gold.fact_pit_stops AS
SELECT
  p.race_id, p.driver_id, p.stop, p.lap, p.stop_time, p.duration,
  ra.year, ra.round, d.full_name AS driver_name
FROM f1_project.silver.pit_stops_clean p
LEFT JOIN f1_project.gold.dim_races ra ON p.race_id = ra.race_id
LEFT JOIN f1_project.gold.dim_drivers d ON p.driver_id = d.driver_id;

CREATE OR REPLACE TABLE f1_project.gold.fact_qualifying AS
SELECT
  q.qualify_id, q.race_id, q.driver_id, q.constructor_id, q.number, q.position,
  q.q1, q.q2, q.q3, ra.year, ra.round, d.full_name AS driver_name
FROM f1_project.silver.qualifying_clean q
LEFT JOIN f1_project.gold.dim_races ra ON q.race_id = ra.race_id
LEFT JOIN f1_project.gold.dim_drivers d ON q.driver_id = d.driver_id;

-- ============ ENRICHED BUSINESS AGGREGATES ============

-- Driver season standings — running total of points, using a window function
CREATE OR REPLACE TABLE f1_project.gold.driver_season_standings AS
SELECT
  driver_id, driver_name, year, round, race_name,
  points AS race_points,
  SUM(points) OVER (
    PARTITION BY driver_id, year ORDER BY round
    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
  ) AS cumulative_points,
  RANK() OVER (PARTITION BY year, round ORDER BY
    SUM(points) OVER (PARTITION BY driver_id, year ORDER BY round
      ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) DESC
  ) AS standing_after_round
FROM f1_project.gold.fact_results;

-- Constructor season standings
CREATE OR REPLACE TABLE f1_project.gold.constructor_season_standings AS
SELECT
  constructor_id, constructor_name, year, round,
  SUM(points) AS round_points,
  SUM(SUM(points)) OVER (
    PARTITION BY constructor_id, year ORDER BY round
    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
  ) AS cumulative_points
FROM f1_project.gold.fact_results
GROUP BY constructor_id, constructor_name, year, round;

-- Pit stop efficiency by driver (enrichment: avg/min/max duration, stop count)
CREATE OR REPLACE TABLE f1_project.gold.driver_pit_stop_efficiency AS
SELECT
  driver_id, driver_name, year,
  COUNT(*) AS total_stops,
  ROUND(AVG(duration), 3) AS avg_stop_duration,
  MIN(duration) AS fastest_stop,
  MAX(duration) AS slowest_stop
FROM f1_project.gold.fact_pit_stops
GROUP BY driver_id, driver_name, year;

-- Race summary view — always-live, good for dashboards
CREATE OR REPLACE VIEW f1_project.gold.race_summary AS
SELECT
  race_id, year, round, race_name, race_date,
  COUNT(DISTINCT driver_id)                                   AS drivers_finished,
  SUM(CASE WHEN status = 'Finished' THEN 1 ELSE 0 END)        AS total_finishers,
  SUM(CASE WHEN status IN ('Retired','DNF') THEN 1 ELSE 0 END) AS total_dnf
FROM f1_project.gold.fact_results
GROUP BY race_id, year, round, race_name, race_date
ORDER BY year, round;

-- ============ Performance tuning ============
-- ZORDER co-locates rows commonly filtered/joined together (driver_id, race_id)
-- so Databricks reads fewer files for typical queries on this fact table.
OPTIMIZE f1_project.gold.fact_results ZORDER BY (driver_id, race_id);

-- ============ Sanity checks ============
SELECT * FROM f1_project.gold.driver_season_standings WHERE year = 2024 AND round = 10 ORDER BY standing_after_round LIMIT 10;
SELECT * FROM f1_project.gold.constructor_season_standings WHERE year = 2024 AND round = 10 ORDER BY cumulative_points DESC;
SELECT * FROM f1_project.gold.driver_pit_stop_efficiency ORDER BY avg_stop_duration LIMIT 10;
SELECT * FROM f1_project.gold.race_summary;
